# SV2 / Edge tuần 10

Mở notebook bằng JupyterLab trên PC và chọn kernel có Matplotlib. Chạy từng ô bằng Shift+Enter. Các ô kiểm checksum và đọc export Git.

In [ ]:
from pathlib import Path
import csv
import hashlib
import json

root = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'results/week10/edge').is_dir())
data = root / 'results/week10/edge'
for line in (data / 'sha256sum_export.txt').read_text().splitlines():
    sha, name = line.split('  ', 1)
    assert hashlib.sha256((data / name).read_bytes()).hexdigest() == sha, name
summary = json.loads((data / 'summary.json').read_text())
lab = json.loads((data / 'lab_summary.json').read_text())
print(summary['run_id'], summary['timing_observation'])
display(summary['warm_switch'])


In [ ]:
from IPython.display import Markdown
table = ['| Lượt | OK / dự kiến | Khớp bit | Trạng thái Edge |',
         '|---|---:|---:|---|']
for r in lab:
    table.append(f"| {r['capture_id']} | {r['ok_count']}/{r['expected_count']} | "
                 f"{r['offline_bit_exact_count']} | {r['edge_review_status']} |")
display(Markdown('\n'.join(table)))


In [ ]:
import matplotlib.pyplot as plt
with (data / 'warm_switch_samples.csv').open(newline='') as stream:
    samples = sorted(float(row['switch_ms']) for row in csv.DictReader(stream))
assert len(samples) == summary['warm_switch']['count']
fig, ax = plt.subplots(figsize=(7, 3.6))
ax.step(samples, [(i + 1) / len(samples) for i in range(len(samples))], where='post', label='Offline')
ax.axvline(50, color='tab:red', linestyle='--', label='Ngưỡng 50 ms')
ax.set(xscale='log', xlabel='Thời gian chuyển split (ms)', ylabel='Tỷ lệ tích lũy')
ax.legend()
fig.tight_layout()
